In [ ]:
import matplotlib.pyplot as plt
import numexpr as ne
import os
import numpy as np
import pandas as pd
import shutil
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mounted at /content/drive


I generated the fractal curves using the alghoritm decribe in the article 'A Fast Algorithm for Simulating the Chordal Schramm–Loewner Evolution' by Tom Kennedy.

In [ ]:
def mappa(z, D_k, d_k):
    return  ne.evaluate("1j * sqrt(4 * D_k - (z)**2) + d_k")

def curve(t, u):
    n = len(t)
    z = np.zeros(n, dtype=np.complex128)
    for i in range(n-1 , 0, -1):
        dt = t[i] - t[i - 1]
        du = u[i] - u[i - 1]
        z[i:] = mappa(z[i:], dt, du)
    return z

def driver(t, kappa=1.0):
    D_k = np.ediff1d(t, to_begin=t[0])
    D_k[0] = 1/len(t)
    d_k = np.random.randn(len(t)) * np.sqrt(D_k * kappa)
    u = np.cumsum(d_k)
    u = u - t*u[-1]
    return u

def curve_generator():
    kappa = np.random.uniform(0, 3.5)
    #np.random.seed(123)
    t = np.linspace(0.0, 1.0, 20000)
    U = driver(t, kappa)
    z = curve(t, U)
    Dim_Fract = 1 + kappa/8

    return z.real, z.imag, Dim_Fract

Example of Creation of a Dataset

In [ ]:
N = 3000  # Number of elements

folder = "DATASET"

os.makedirs(folder, exist_ok=True)

label = []
path = []

for i in range(N):

    x, y, fractal_dimension = curve_generator()

    # Stack the coordinates into an (N×2) array
    coords = np.column_stack((x, y))

    # Save the coordinates as .npy file
    data_path = os.path.join(folder, f"curve_{i}.npy")
    np.save(data_path, coords)

    # Save the label and the path (on Drive)
    label.append(fractal_dimension)
    path.append(os.path.join("/content/drive/My Drive/", folder, f"curve_{i}.npy"))

# Save the labels into a summary CSV file
df_label = pd.DataFrame({
    "Path_Coordinates": folder,
    "Fractal_Dimension": label
})
df_label.to_csv(os.path.join(folder, "a_etichette.csv"), index=False)


path_dri = os.path.join("/content/drive/My Drive/", folder)

# Copy the folder into Drive
shutil.copytree(folder, path_dri)

'/content/drive/My Drive/DATASET-7'